# NB0: Setup & Translation Verification

**Project 1**: Language as a Hidden Variable: Measuring Behavioral Divergence in Multilingual LLMs

## Purpose
- Install and verify all required libraries
- Verify Groq API access
- Load and validate the prompt master CSV (30 prompts × 3 languages)
- Compute BLEU scores for back-translations
- Document translation quality

## Prerequisites
- `prompts_master.csv` must be fully filled (English, Hindi, French, back-translations)
- Groq API key stored in Colab Secrets as `GROQ_API_KEY`

## Outputs
- `bleu_scores.csv`
- Validated `prompts_master.csv`

---
## [0.1] Install and Import Libraries

In [1]:
# Install required packages
!pip install -q groq sentence-transformers transformers torch nltk sacrebleu pandas numpy scipy scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import os
import time
import json
from groq import Groq

nltk.download('punkt')
nltk.download('punkt_tab')

print("All libraries imported successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All libraries imported successfully.


---
## [0.2] API Key Setup

In [3]:
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


---
## [0.3] Test API Connection
Send one test prompt to verify the API is working.

In [4]:
# Test with both models
# models = ["llama-3.1-70b-versatile", "mixtral-8x7b-32768"]
# models = [
#     "llama-3.3-70b-versatile",
#     "llama-3.1-8b-instant"   # very fast, cheap
# ]
models = [
    "llama-3.3-70b-versatile",   # updated Llama
    "openai/gpt-oss-20b",     # newer Mixtral variant
]
test_prompt = "What is 2 + 2? Answer in one sentence."

for model_name in models:
    start_time = time.time()
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": test_prompt}],
            temperature=0.1,
            max_tokens=512,
            top_p=1.0
        )
        latency = (time.time() - start_time) * 1000
        reply = response.choices[0].message.content
        print(f"\n--- Model: {model_name} ---")
        print(f"Response: {reply}")
        print(f"Response length: {len(reply)} chars")
        print(f"Latency: {latency:.0f} ms")
        print("STATUS: OK")
    except Exception as e:
        print(f"\n--- Model: {model_name} ---")
        print(f"ERROR: {e}")
    time.sleep(2)


--- Model: llama-3.3-70b-versatile ---
Response: The answer to 2 + 2 is 4.
Response length: 25 chars
Latency: 225 ms
STATUS: OK

--- Model: openai/gpt-oss-20b ---
Response: 2 + 2 equals 4.
Response length: 15 chars
Latency: 143 ms
STATUS: OK


---
## [0.4] Load and Validate Prompt Master CSV

Upload `prompts_master.csv` to the Colab file browser, or mount Google Drive.

In [5]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Set the project data directory
# UPDATE THIS PATH to match your Google Drive folder structure
DATA_DIR = '/content/drive/MyDrive/nlp_genai_cie3_2/data'
RESULTS_DIR = '/content/drive/MyDrive/nlp_genai_cie3_2/results'
FIGURES_DIR = '/content/drive/MyDrive/nlp_genai_cie3_2/results/figures'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Mounted at /content/drive
Data directory: /content/drive/MyDrive/nlp_genai_cie3_2/data
Results directory: /content/drive/MyDrive/nlp_genai_cie3_2/results


In [9]:
# Load and validate prompts_master.csv
prompts_path = os.path.join(DATA_DIR, 'prompts_master_csv_utf.csv')
# df_prompts = pd.read_csv(prompts_path)
df_prompts = pd.read_csv(prompts_path)

print("=== VALIDATION CHECKS ===")

# Check 1: Exactly 30 rows
assert len(df_prompts) == 30, f"Expected 30 rows, got {len(df_prompts)}"
print(f"[PASS] Row count: {len(df_prompts)}")

# Check 2: Required columns present
required_cols = ['prompt_id', 'category', 'english', 'hindi', 'french']
for col in required_cols:
    assert col in df_prompts.columns, f"Missing column: {col}"
print(f"[PASS] All required columns present: {required_cols}")

# Check 3: No empty cells in required columns
for col in required_cols:
    null_count = df_prompts[col].isnull().sum()
    empty_count = (df_prompts[col].astype(str).str.strip() == '').sum()
    total_missing = null_count + empty_count
    if total_missing > 0:
        print(f"[WARNING] Column '{col}' has {total_missing} empty cells")
        if col in ['hindi', 'french']:
            print(f"  -> Fill translations before running NB1!")
    else:
        print(f"[PASS] Column '{col}': no empty cells")

# Check 4: Category distribution (10 each)
cat_counts = df_prompts['category'].value_counts()
print(f"\nCategory distribution:")
for cat, count in cat_counts.items():
    status = 'PASS' if count == 10 else 'FAIL'
    print(f"  [{status}] {cat}: {count} prompts")

# Display the dataframe
print("\n=== PROMPT PREVIEW ===")
df_prompts[['prompt_id', 'category', 'hindi']].head(10)

=== VALIDATION CHECKS ===
[PASS] Row count: 30
[PASS] All required columns present: ['prompt_id', 'category', 'english', 'hindi', 'french']
[PASS] Column 'prompt_id': no empty cells
[PASS] Column 'category': no empty cells
[PASS] Column 'english': no empty cells
[PASS] Column 'hindi': no empty cells
[PASS] Column 'french': no empty cells

Category distribution:
  [PASS] FACTUAL: 10 prompts
  [PASS] NORMATIVE: 10 prompts
  [PASS] SAFETY: 10 prompts

=== PROMPT PREVIEW ===


,prompt_id,category,hindi
0,F01,FACTUAL,किसी अर्थव्यवस्था में मुद्रास्फीति का कारण क्य...
1,F02,FACTUAL,मानव प्रतिरक्षा प्रणाली वायरस से कैसे लड़ती है?
2,F03,FACTUAL,ग्रीनहाउस प्रभाव क्या है और यह जलवायु परिवर्तन...
3,F04,FACTUAL,टीके बीमारियों को रोकने के लिए कैसे काम करते हैं?
4,F05,FACTUAL,नाभिकीय विखंडन और नाभिकीय संलयन में क्या अंतर है?
5,F06,FACTUAL,इंटरनेट दुनिया भर में डेटा कैसे प्रसारित करता है?
6,F07,FACTUAL,भूकंप किस कारण से आते हैं?
7,F08,FACTUAL,पौधे प्रकाश संश्लेषण के माध्यम से ऊर्जा कैसे उ...
8,F09,FACTUAL,किसी देश की अर्थव्यवस्था में केंद्रीय बैंक की ...
9,F10,FACTUAL,डीएनए आनुवंशिक जानकारी कैसे वहन करता है?


In [11]:
# Display full prompt table
pd.set_option('display.max_colwidth', 80)
df_prompts[['prompt_id', 'category', 'hindi']]

,prompt_id,category,hindi
0,F01,FACTUAL,किसी अर्थव्यवस्था में मुद्रास्फीति का कारण क्या होता है?
1,F02,FACTUAL,मानव प्रतिरक्षा प्रणाली वायरस से कैसे लड़ती है?
2,F03,FACTUAL,ग्रीनहाउस प्रभाव क्या है और यह जलवायु परिवर्तन का कारण कैसे बनता है?
3,F04,FACTUAL,टीके बीमारियों को रोकने के लिए कैसे काम करते हैं?
4,F05,FACTUAL,नाभिकीय विखंडन और नाभिकीय संलयन में क्या अंतर है?
5,F06,FACTUAL,इंटरनेट दुनिया भर में डेटा कैसे प्रसारित करता है?
6,F07,FACTUAL,भूकंप किस कारण से आते हैं?
7,F08,FACTUAL,पौधे प्रकाश संश्लेषण के माध्यम से ऊर्जा कैसे उत्पन्न करते हैं?
8,F09,FACTUAL,किसी देश की अर्थव्यवस्था में केंद्रीय बैंक की क्या भूमिका होती है?
9,F10,FACTUAL,डीएनए आनुवंशिक जानकारी कैसे वहन करता है?


---
## [0.5] BLEU Score Computation

Compute BLEU scores comparing original English prompts with back-translated versions.

**Requires**: `hindi_back_translated` and `french_back_translated` columns filled in `prompts_master.csv`.

Flag any prompt where BLEU < 0.6.

In [13]:
from nltk.tokenize import word_tokenize

smoothing = SmoothingFunction().method1

bleu_results = []

for _, row in df_prompts.iterrows():
    prompt_id = row['prompt_id']
    category = row['category']
    english_original = str(row['english']).strip()

    # Hindi back-translation BLEU
    hindi_bt = str(row.get('hindi_back_translated', '')).strip()
    if hindi_bt and hindi_bt != 'nan':
        ref_tokens = word_tokenize(english_original.lower())
        hyp_tokens = word_tokenize(hindi_bt.lower())
        hindi_bleu = sentence_bleu(
            [ref_tokens], hyp_tokens,
            smoothing_function=smoothing
        )
    else:
        hindi_bleu = None

    # French back-translation BLEU
    french_bt = str(row.get('french_back_translated', '')).strip()
    if french_bt and french_bt != 'nan':
        ref_tokens = word_tokenize(english_original.lower())
        hyp_tokens = word_tokenize(french_bt.lower())
        french_bleu = sentence_bleu(
            [ref_tokens], hyp_tokens,
            smoothing_function=smoothing
        )
    else:
        french_bleu = None

    bleu_results.append({
        'prompt_id': prompt_id,
        'category': category,
        'hindi_bleu': round(hindi_bleu, 4) if hindi_bleu is not None else None,
        'french_bleu': round(french_bleu, 4) if french_bleu is not None else None,
        'hindi_flagged': (hindi_bleu is not None and hindi_bleu < 0.6),
        'french_flagged': (french_bleu is not None and french_bleu < 0.6)
    })

df_bleu = pd.DataFrame(bleu_results)

# Display results
print("=== BLEU SCORES ===")
print(df_bleu.to_string(index=False))

# Summary
print(f"\n=== SUMMARY ===")
if df_bleu['hindi_bleu'].notna().any():
    print(f"Mean Hindi BLEU: {df_bleu['hindi_bleu'].mean():.4f}")
    flagged_hi = df_bleu[df_bleu['hindi_flagged'] == True]
    print(f"Hindi flagged (BLEU < 0.6): {len(flagged_hi)} prompts")
    if len(flagged_hi) > 0:
        print(f"  Flagged IDs: {flagged_hi['prompt_id'].tolist()}")
else:
    print("Hindi back-translations not yet available.")

if df_bleu['french_bleu'].notna().any():
    print(f"Mean French BLEU: {df_bleu['french_bleu'].mean():.4f}")
    flagged_fr = df_bleu[df_bleu['french_flagged'] == True]
    print(f"French flagged (BLEU < 0.6): {len(flagged_fr)} prompts")
    if len(flagged_fr) > 0:
        print(f"  Flagged IDs: {flagged_fr['prompt_id'].tolist()}")
else:
    print("French back-translations not yet available.")

=== BLEU SCORES ===
prompt_id  category  hindi_bleu  french_bleu  hindi_flagged  french_flagged
      F01   FACTUAL      1.0000       1.0000          False           False
      F02   FACTUAL      1.0000       1.0000          False           False
      F03   FACTUAL      1.0000       1.0000          False           False
      F04   FACTUAL      0.7071       0.7071          False           False
      F05   FACTUAL      1.0000       1.0000          False           False
      F06   FACTUAL      0.6580       1.0000          False           False
      F07   FACTUAL      1.0000       0.0831          False            True
      F08   FACTUAL      1.0000       1.0000          False           False
      F09   FACTUAL      1.0000       1.0000          False           False
      F10   FACTUAL      1.0000       1.0000          False           False
      N01 NORMATIVE      0.6781       0.6781          False           False
      N02 NORMATIVE      0.8003       0.8003          False         

In [14]:
# Save BLEU scores
bleu_path = os.path.join(DATA_DIR, 'bleu_scores.csv')
df_bleu.to_csv(bleu_path, index=False)
print(f"Saved: {bleu_path}")

Saved: /content/drive/MyDrive/nlp_genai_cie3_2/data/bleu_scores.csv


---
## [0.6] Manual Inspection Log

Generate a template for translation inspection notes.
**Team members** should fill this in after reviewing all translations.

In [15]:
# Generate translation inspection template
notes_path = os.path.join(DATA_DIR, 'translation_inspection_notes.txt')

lines = [
    "=" * 80,
    "TRANSLATION INSPECTION NOTES",
    "Project 1: Language as a Hidden Variable",
    f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 80,
    "",
    "Instructions:",
    "For each prompt, inspect Hindi and French translations.",
    "Note any meaning shifts, ambiguities, or cultural mismatches.",
    "This file will be cited in the paper methodology section.",
    "",
    "Format: PROMPT_ID | LANGUAGE | BLEU | OBSERVATION",
    "-" * 80,
    ""
]

for _, row in df_prompts.iterrows():
    pid = row['prompt_id']
    bleu_row = df_bleu[df_bleu['prompt_id'] == pid].iloc[0]
    hi_bleu = bleu_row['hindi_bleu'] if pd.notna(bleu_row['hindi_bleu']) else 'N/A'
    fr_bleu = bleu_row['french_bleu'] if pd.notna(bleu_row['french_bleu']) else 'N/A'
    hi_flag = ' [FLAGGED]' if bleu_row['hindi_flagged'] else ''
    fr_flag = ' [FLAGGED]' if bleu_row['french_flagged'] else ''

    lines.append(f"{pid} | English: {row['english']}")
    lines.append(f"{pid} | Hindi  | BLEU={hi_bleu}{hi_flag} | Notes: ")
    lines.append(f"{pid} | French | BLEU={fr_bleu}{fr_flag} | Notes: ")
    lines.append("")

with open(notes_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f"Saved: {notes_path}")
print("\nPlease fill in the 'Notes:' fields for each prompt after manual review.")

Saved: /content/drive/MyDrive/nlp_genai_cie3_2/data/translation_inspection_notes.txt

Please fill in the 'Notes:' fields for each prompt after manual review.


---
## Summary

### Checklist before proceeding to NB1:
- [ ] All libraries installed without error
- [ ] Both models respond to API test call
- [ ] `prompts_master.csv` has 30 rows, no empty cells in required columns
- [ ] BLEU scores computed for all back-translations
- [ ] `bleu_scores.csv` saved
- [ ] Flagged prompts documented
- [ ] `translation_inspection_notes.txt` filled by team

### Output files:
- `bleu_scores.csv`
- `translation_inspection_notes.txt`